# DOWNLOAD CÁC THƯ VIỆN VÀ DATASET

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets seqeval accelerate gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 36.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import copy
import logging
import shutil
import torch
import re
import gradio as gr
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset as HFDataset
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
!gdown 1DuwYRftQjQmQcAR4FVPH-HvGuxGi4ist
!gdown 11xZZfla8CDH54-EeUUdnAAoT2ummuEJh
!gdown 1wVyhQkhAzwod2at7Ir3tRHbdCMOszzwg

Downloading...
From: https://drive.google.com/uc?id=1DuwYRftQjQmQcAR4FVPH-HvGuxGi4ist
To: /content/train_word.conll
100% 1.42M/1.42M [00:00<00:00, 113MB/s]
Downloading...
From: https://drive.google.com/uc?id=11xZZfla8CDH54-EeUUdnAAoT2ummuEJh
To: /content/test_word.conll
100% 958k/958k [00:00<00:00, 36.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wVyhQkhAzwod2at7Ir3tRHbdCMOszzwg
To: /content/dev_word.conll
100% 628k/628k [00:00<00:00, 95.3MB/s]


# 1. TIỀN XỬ LÝ DỮ LIỆU

**1.1 Đọc các file dữ liệu để trainning**

In [ ]:
# Hàm đọc file định dạng CoNLL
def read_conll(file_path):
    sentences, sentence_labels, unique_labels = [], [], set()
    with open(file_path, 'r', encoding='utf-8') as file:
        tokens, labels = [], []
        for line in file:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(' '.join(tokens))
                    sentence_labels.append(' '.join(labels))
                    tokens, labels = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                label = parts[1] if len(parts) > 1 else 'O'
                labels.append(label)
                unique_labels.add(label)
    if tokens:
        sentences.append(' '.join(tokens))
        sentence_labels.append(' '.join(labels))

    print(f"Unique labels found: {unique_labels}")
    return sentences, sentence_labels, sorted(unique_labels)

# Đọc file dữ liệu
train_sentences, train_labels, labels_set = read_conll("./train_word.conll")
dev_sentences, dev_labels, _ = read_conll("./dev_word.conll")
test_sentences, test_labels, _ = read_conll("./test_word.conll")

Unique labels found: {'B-DATE', 'O', 'I-AGE', 'I-TRANSPORTATION', 'I-SYMPTOM_AND_DISEASE', 'I-ORGANIZATION', 'I-NAME', 'B-ORGANIZATION', 'B-TRANSPORTATION', 'B-SYMPTOM_AND_DISEASE', 'B-JOB', 'B-AGE', 'B-PATIENT_ID', 'I-DATE', 'B-GENDER', 'I-JOB', 'B-NAME', 'B-LOCATION', 'I-PATIENT_ID', 'I-LOCATION'}
Unique labels found: {'B-DATE', 'O', 'I-TRANSPORTATION', 'I-SYMPTOM_AND_DISEASE', 'I-ORGANIZATION', 'I-NAME', 'B-ORGANIZATION', 'B-TRANSPORTATION', 'B-SYMPTOM_AND_DISEASE', 'B-JOB', 'B-AGE', 'B-PATIENT_ID', 'I-DATE', 'B-GENDER', 'I-JOB', 'B-NAME', 'B-LOCATION', 'I-PATIENT_ID', 'I-LOCATION'}
Unique labels found: {'B-DATE', 'O', 'I-AGE', 'I-TRANSPORTATION', 'I-SYMPTOM_AND_DISEASE', 'I-ORGANIZATION', 'I-NAME', 'B-ORGANIZATION', 'B-TRANSPORTATION', 'B-SYMPTOM_AND_DISEASE', 'B-JOB', 'B-AGE', 'B-PATIENT_ID', 'I-DATE', 'B-GENDER', 'I-JOB', 'B-NAME', 'B-LOCATION', 'I-PATIENT_ID', 'I-LOCATION'}


In [ ]:
label_list = sorted(list(labels_set))
label_map = {label: i for i, label in enumerate(label_list)}

In [ ]:
print("Label list khi train:")
for idx, label in enumerate(label_list):
    print(f"{idx}: {label}")

# Lưu ra file để dùng lại khi inference
with open("labels.txt", "w", encoding="utf-8") as f:
    for label in label_list:
        f.write(label + "\n")

Label list khi train:
0: B-AGE
1: B-DATE
2: B-GENDER
3: B-JOB
4: B-LOCATION
5: B-NAME
6: B-ORGANIZATION
7: B-PATIENT_ID
8: B-SYMPTOM_AND_DISEASE
9: B-TRANSPORTATION
10: I-AGE
11: I-DATE
12: I-JOB
13: I-LOCATION
14: I-NAME
15: I-ORGANIZATION
16: I-PATIENT_ID
17: I-SYMPTOM_AND_DISEASE
18: I-TRANSPORTATION
19: O


In [ ]:
print("Label List:", label_list)
print("Label Map:", label_map)

Label List: ['B-AGE', 'B-DATE', 'B-GENDER', 'B-JOB', 'B-LOCATION', 'B-NAME', 'B-ORGANIZATION', 'B-PATIENT_ID', 'B-SYMPTOM_AND_DISEASE', 'B-TRANSPORTATION', 'I-AGE', 'I-DATE', 'I-JOB', 'I-LOCATION', 'I-NAME', 'I-ORGANIZATION', 'I-PATIENT_ID', 'I-SYMPTOM_AND_DISEASE', 'I-TRANSPORTATION', 'O']
Label Map: {'B-AGE': 0, 'B-DATE': 1, 'B-GENDER': 2, 'B-JOB': 3, 'B-LOCATION': 4, 'B-NAME': 5, 'B-ORGANIZATION': 6, 'B-PATIENT_ID': 7, 'B-SYMPTOM_AND_DISEASE': 8, 'B-TRANSPORTATION': 9, 'I-AGE': 10, 'I-DATE': 11, 'I-JOB': 12, 'I-LOCATION': 13, 'I-NAME': 14, 'I-ORGANIZATION': 15, 'I-PATIENT_ID': 16, 'I-SYMPTOM_AND_DISEASE': 17, 'I-TRANSPORTATION': 18, 'O': 19}


**1.2 Chuyển đổi dữ liệu thành Dataset của Hugging Face**

In [ ]:
def prepare_dataset(sentences, labels):
    return {'tokens': sentences, 'labels': labels}

def process_string_to_array(dataset):
    return {
        'tokens': [s.split() for s in dataset['tokens']],
        'labels': [l.split() for l in dataset['labels']]
    }

# Chuẩn bị và xử lý dữ liệu
train_dataset = process_string_to_array(prepare_dataset(train_sentences, train_labels))
dev_dataset = process_string_to_array(prepare_dataset(dev_sentences, dev_labels))
test_dataset = process_string_to_array(prepare_dataset(test_sentences, test_labels))

# Chuyển đổi thành Dataset của Hugging Face
train_dataset = HFDataset.from_dict(train_dataset)
dev_dataset = HFDataset.from_dict(dev_dataset)
test_dataset = HFDataset.from_dict(test_dataset)

In [ ]:
# In kích thước của các dataset và mẫu để kiểm tra
print(f"Train dataset size: {len(train_dataset)}")
print("Train dataset sample:", train_dataset[0])

print(f"Dev dataset size: {len(dev_dataset)}")
print("Dev dataset sample:", dev_dataset[0])

print(f"Test dataset size: {len(test_dataset)}")
print("Test dataset sample:", test_dataset[0])

Train dataset size: 5027
Train dataset sample: {'tokens': ['Đồng_thời', ',', 'bệnh_viện', 'tiếp_tục', 'thực_hiện', 'các', 'biện_pháp', 'phòng_chống', 'dịch_bệnh', 'COVID', '-', '19', 'theo', 'hướng_dẫn', 'của', 'Bộ', 'Y_tế', '.'], 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'O']}
Dev dataset size: 2000
Dev dataset sample: {'tokens': ['Bác_sĩ', 'Nguyễn_Trung_Nguyên', ',', 'Giám_đốc', 'Trung_tâm', 'Chống', 'độc', ',', 'Bệnh_viện', 'Bạch_Mai', ',', 'cho', 'biết', 'bệnh_nhân', 'được', 'chuyển', 'đến', 'bệnh_viện', 'ngày', '7/3', ',', 'chẩn_đoán', 'ngộ_độc', 'thuốc', 'điều_trị', 'sốt_rét', 'chloroquine', '.'], 'labels': ['O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'I-ORGANIZATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DATE', 'O', 'O', 'B-SYMPTOM_AND_DISEASE', 'I-SYMPTOM_AND_DISEASE', 'O', 'O', 'O', 'O']}
Test dataset size: 3000
Test datas

**1.3 Chuyển dữ liệu thành Features để huấn luyện**

  1.3.1 Chuyển dữ liệu thô từ dictionary thành danh sách các đối tượng Example, giúp dễ xử lý hơn sau này.

In [ ]:
class Example:
    def __init__(self, words, slot_labels, guid=None):
        self.words = words
        self.slot_labels = slot_labels
        self.guid = guid

def convert_dataset_to_examples(dataset):
    return [Example(words=tokens, slot_labels=labels, guid=i)
            for i, (tokens, labels) in enumerate(zip(dataset['tokens'], dataset['labels']))]

# Chuyển dữ liệu thành Example
train_examples = convert_dataset_to_examples(train_dataset)
dev_examples = convert_dataset_to_examples(dev_dataset)
test_examples = convert_dataset_to_examples(test_dataset)

In [ ]:
print(train_examples[0].words)
print(train_examples[0].slot_labels)

['Đồng_thời', ',', 'bệnh_viện', 'tiếp_tục', 'thực_hiện', 'các', 'biện_pháp', 'phòng_chống', 'dịch_bệnh', 'COVID', '-', '19', 'theo', 'hướng_dẫn', 'của', 'Bộ', 'Y_tế', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'O']


1.3.2 Chuyển dữ liệu từ Example thành Features để có thể đưa vào mô hình

In [ ]:
# Lớp InputFeatures
class InputFeatures:
    def __init__(self, input_ids, attention_mask, token_type_ids, slot_labels_ids):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.token_type_ids = token_type_ids
        self.slot_labels_ids = slot_labels_ids

In [ ]:
# Hàm chuyển từ đối tượng Example sang Features
def convert_examples_to_features(examples, max_seq_len, tokenizer, label_map, pad_label_id=-100):
    features = []
    for example in examples:
        tokens, label_ids = [], []
        for word, label in zip(example.words, example.slot_labels):
            word_tokens = tokenizer.tokenize(word) or [tokenizer.unk_token]
            tokens.extend(word_tokens)
            label_ids.extend([label_map[label]] + [pad_label_id] * (len(word_tokens) - 1))
        if len(tokens) > max_seq_len - 2:
            tokens, label_ids = tokens[:max_seq_len - 2], label_ids[:max_seq_len - 2]
        tokens = [tokenizer.cls_token] + tokens + [tokenizer.sep_token]
        label_ids = [pad_label_id] + label_ids + [pad_label_id]
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)
        padding_length = max_seq_len - len(input_ids)
        input_ids += [tokenizer.pad_token_id] * padding_length
        attention_mask += [0] * padding_length
        label_ids += [pad_label_id] * padding_length
        features.append(InputFeatures(input_ids, attention_mask, None, label_ids))
    return features

In [ ]:
# Khởi tạo tokenizer của PhoBERT
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base', use_fast=True)
tokenizer.add_prefix_space = True
max_seq_len = 128

# Chuyển examples sang features
train_features = convert_examples_to_features(train_examples, max_seq_len, tokenizer, label_map)
dev_features = convert_examples_to_features(dev_examples, max_seq_len, tokenizer, label_map)
test_features = convert_examples_to_features(test_examples, max_seq_len, tokenizer, label_map)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
print(train_features[0].input_ids)
print(train_features[0].attention_mask)
print(train_features[0].slot_labels_ids)

[0, 1248, 4, 757, 194, 112, 9, 717, 2137, 3795, 9089, 6232, 1927, 31, 1195, 63, 1010, 7, 125, 1059, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[-100, 19, 19, 19, 19, 19, 19, 19, 19, 19, 19, -100, -100, 19, 19, 19, 19, 19, 6, 15, 19, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,

1.3.3 Tạo Dataset từ các Features đã chuẩn hóa

In [ ]:
class NERDataset(Dataset):
    def __init__(self, features):
        self.features = features
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        feature = self.features[idx]
        return {
            'input_ids': torch.tensor(feature.input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(feature.attention_mask, dtype=torch.long),
            'labels': torch.tensor(feature.slot_labels_ids, dtype=torch.long),
        }

# Chuyển features thành dataset dạng torch cho huấn luyện và đánh giá
train_dataset = NERDataset(train_features)
dev_dataset = NERDataset(dev_features)
test_dataset = NERDataset(test_features)

In [ ]:
train_dataset[0]

{'input_ids': tensor([   0, 1248,    4,  757,  194,  112,    9,  717, 2137, 3795, 9089, 6232,
         1927,   31, 1195,   63, 1010,    7,  125, 1059,    5,    2,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
         0, 0, 0, 0

# 2. HUẤN LUYỆN MÔ HÌNH
---
**- Khởi tạo mô hình PhoBERT**<br>
**- Thiết lập hàm đánh giá "compute_metrics()"**<br>
**- Cấu hình các tham số huấn luyện**<br>
**- Huấn luyện**<br>
**- Đánh giá trên tập "dev_dataset**

In [ ]:
# Tự động đăng nhập wandb
os.environ["WANDB_API_KEY"] = "fb80f73fcb020dd331c4509a5851a96be928f490"  # Thêm mã API của bạn ở đây

# Tải tokenizer và model PhoBERT
model = AutoModelForTokenClassification.from_pretrained("vinai/phobert-base", num_labels=len(label_list))

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    labels = p.label_ids
    true_preds, true_labels = [], []

    for pred, label in zip(preds, labels):
        tmp_preds, tmp_labels = [], []
        for p_i, l_i in zip(pred, label):
            if l_i != -100:
                tmp_preds.append(label_list[p_i])
                tmp_labels.append(label_list[l_i])
        true_preds.append(tmp_preds)
        true_labels.append(tmp_labels)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

# Cấu hình huấn luyện
training_args = TrainingArguments(
    output_dir="./ner_phobert",
    do_train=True,
    do_eval=True,  # Bật việc đánh giá
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    report_to=["wandb"],
)

# Huấn luyện mô hình
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  # Dataset huấn luyện
    eval_dataset=dev_dataset,  # Dataset đánh giá
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Huấn luyện và đánh giá
trainer.train()
eval_metrics = trainer.evaluate()
print(eval_metrics)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-20-231a3278b7cf>:43: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: vkhlinh (vkhlinh-l) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,1.773000
20,0.951600
30,0.678500
40,0.515800
50,0.448800
60,0.334000
70,0.296200
80,0.196800
90,0.199600
100,0.165100


{'eval_loss': 0.06559909135103226, 'eval_precision': 0.9495776135163675, 'eval_recall': 0.9636972538513061, 'eval_f1': 0.9565853334219799, 'eval_runtime': 14.3998, 'eval_samples_per_second': 138.891, 'eval_steps_per_second': 8.681, 'epoch': 3.0}


# 3. ĐÁNH GIÁ MÔ HÌNH

**3.1 Đánh giá trên tập "test_dataset"**

In [ ]:
# Đánh giá mô hình trên test set
test_metrics = trainer.evaluate(test_dataset)

# In kết quả
print("\n====== Đánh giá trên TEST set ======")
print(f"Loss: {test_metrics['eval_loss']:.4f}")
print(f"Precision: {test_metrics['eval_precision']:.4f}")
print(f"Recall: {test_metrics['eval_recall']:.4f}")
print(f"F1 Score: {test_metrics['eval_f1']:.4f}")


====== Đánh giá trên TEST set ======
Loss: 0.0748
Precision: 0.9357
Recall: 0.9513
F1 Score: 0.9434


**3.2 Đánh giá theo từng loại nhãn**

In [ ]:
# Lấy nhãn thực và nhãn dự đoán từ tập test
predictions, labels, _ = trainer.predict(test_dataset)
preds = predictions.argmax(-1)  # Chọn nhãn có xác suất cao nhất từ predictions

true_labels = []
true_preds = []

# Duyệt qua từng cặp dự đoán và nhãn thực tế
for pred, label in zip(preds, labels):
    true_label = []
    true_pred = []
    # Duyệt qua từng cặp nhãn trong token
    for p_i, l_i in zip(pred, label):
        if l_i != -100:  # Lọc các nhãn không hợp lệ (-100 là nhãn bỏ qua)
            true_label.append(label_list[l_i])  # Lấy nhãn thực tế
            true_pred.append(label_list[p_i])   # Lấy nhãn dự đoán
    true_labels.append(true_label)
    true_preds.append(true_pred)

# In báo cáo chi tiết
print("\n====== Báo cáo chi tiết theo từng thực thể ======")
print(classification_report(true_labels, true_preds, digits=4))



====== Báo cáo chi tiết theo từng thực thể ======
                     precision    recall  f1-score   support

                AGE     0.9637    0.9620    0.9628       579
               DATE     0.9762    0.9921    0.9841      1652
             GENDER     0.9678    0.9804    0.9741       460
                JOB     0.6878    0.7514    0.7182       173
           LOCATION     0.9358    0.9526    0.9441      4435
               NAME     0.8825    0.9214    0.9015       318
       ORGANIZATION     0.8681    0.8962    0.8819       771
         PATIENT_ID     0.9776    0.9860    0.9818      1995
SYMPTOM_AND_DISEASE     0.8713    0.8820    0.8766      1136
     TRANSPORTATION     0.9791    0.9689    0.9740       193

          micro avg     0.9357    0.9513    0.9434     11712
          macro avg     0.9110    0.9293    0.9199     11712
       weighted avg     0.9361    0.9513    0.9436     11712




**3.3 Lưu mô hình và tokenizer sau khi train**

In [ ]:
save_directory = '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned'

# Lưu mô hình
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

('/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned/vocab.txt',
 '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned/bpe.codes',
 '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned/added_tokens.json')

# 4. Dùng mô hình sau huấn luyện để nhận diện thực thể trong đoạn văn

In [ ]:
def split_sentences(text):
    # Tách câu theo dấu câu, giữ lại dấu câu ở cuối
    sentences = re.split(r'(?<=[.!?…])\s+', text.strip())
    # Loại bỏ câu rỗng
    sentences = [s for s in sentences if s.strip()]
    return sentences


# Tải mô hình và tokenizer đã huấn luyện
model_dir = '/content/drive/MyDrive/Ner_PhoBERT_finetuned/phobert-ner-finetuned'

model = AutoModelForTokenClassification.from_pretrained(model_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)

id2label = {i: label for i, label in enumerate(label_list)}

def predict_entities_for_text(text):
    sentences = split_sentences(text)
    print(f"Phát hiện {len(sentences)} câu trong đoạn văn.")


    for idx, sent in enumerate(sentences, 1):
        print(f"\n------ Câu {idx} ------")
        # Tokenize
        inputs = tokenizer(sent, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits

        predictions = torch.argmax(logits, dim=-1)
        predicted_labels = [id2label[label.item()] for label in predictions[0]]
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        print(f"{'Token':<15} {'Predicted Label'}")
        print("-" * 40)
        for token, label in zip(tokens, predicted_labels):
            print(f"{token:<15} {label}")
text = input("Nhập đoạn văn để nhận dạng thực thể: ")
predict_entities_for_text(text)

Nhập đoạn văn để nhận dạng thực thể: Hôm nay, ngày 8 tháng 6 năm 2025, Nguyễn Văn An, một kỹ sư phần mềm, đã tham gia hội thảo công nghệ tại Hà Nội. Anh ấy làm việc cho công ty Công nghệ FPT, một trong những tập đoàn lớn nhất Việt Nam. Hội thảo được tổ chức tại Trung tâm Hội nghị Quốc gia, nơi thu hút hơn 500 chuyên gia từ khắp nơi. An đã gặp Trần Thị Bình, một nhà nghiên cứu AI đến từ Đại học Bách Khoa. Họ cùng thảo luận về dự án trí tuệ nhân tạo với Google
Phát hiện 5 câu trong đoạn văn.

------ Câu 1 ------
Token           Predicted Label
----------------------------------------
<s>             O
Hôm             O
nay@@           O
,               O
ngày            O
8               B-DATE
tháng           I-DATE
6               I-DATE
năm             O
20@@            I-DATE
25@@            I-DATE
,               O
Nguyễn          B-NAME
Văn             B-NAME
An@@            B-NAME
,               O
một             O
kỹ              O
sư              O
phần            O
m@@        